# Step 0 — SAM zero-shot su Kvasir-SEG (Kaggle)

Obiettivo: capire in pratica come funziona SAM prima di scrivere la repo.
Alla fine di questo notebook avrai:
1. caricato un'immagine e la sua maschera,
2. calcolato il box prompt dalla maschera,
3. ottenuto una predizione di SAM e il suo Dice,
4. un primo Dice medio su 20 immagini (non è il numero ufficiale: niente split di test ancora).

**Impostazioni Kaggle (pannello a destra → Session options):**
- *Accelerator*: **GPU T4 x2** (il codice ne usa una sola) oppure P100.
- *Internet*: **On** (serve per `pip install` e per scaricare i pesi; richiede il telefono verificato sull'account).
- *Persistence*: "Files only" ti evita di riscaricare dati e pesi a ogni sessione.

## 1. Controllo GPU

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), "Accendi la GPU in Session options → Accelerator"
print(torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Installazione di `segment-anything`
PyTorch, OpenCV e matplotlib sono già presenti nell'immagine Kaggle.

In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

## 3. Dati e pesi

- `/kaggle/input/` è **sola lettura**: lì finiscono i dataset aggiunti con *Add Input*.
- `/kaggle/working/` è la cartella scrivibile (20 GB), salvata quando fai *Save Version*.

Se hai aggiunto Kvasir-SEG come input Kaggle, la cella lo trova da sola; altrimenti lo scarica dal sito ufficiale.

In [ ]:
from pathlib import Path

WORK = Path("/kaggle/working")
CKPT = WORK / "checkpoints" / "sam_vit_b_01ec64.pth"

def find_kvasir_root():
    base = Path("/kaggle/input")
    if base.exists():
        for p in base.rglob("images"):
            if (p.parent / "masks").is_dir():
                return p.parent
    local = WORK / "data" / "Kvasir-SEG"
    return local if local.exists() else None

root = find_kvasir_root()
if root is None:
    (WORK / "data").mkdir(parents=True, exist_ok=True)
    !wget -q https://datasets.simula.no/downloads/kvasir-seg.zip -O /kaggle/working/data/kvasir-seg.zip
    !unzip -q /kaggle/working/data/kvasir-seg.zip -d /kaggle/working/data/
    !rm /kaggle/working/data/kvasir-seg.zip
    root = find_kvasir_root()

if not CKPT.exists():
    CKPT.parent.mkdir(parents=True, exist_ok=True)
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -P /kaggle/working/checkpoints/

ids = sorted(p.stem for p in (root / "images").glob("*.jpg"))
print(root, "| immagini:", len(ids), "| checkpoint:", CKPT.exists())

## 4. Un'immagine e la sua maschera

Le maschere sono JPEG: la compressione crea valori intermedi (3, 128, 250...), quindi le binarizziamo con `> 127`.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def load_sample(img_id):
    img = cv2.cvtColor(cv2.imread(str(root / "images" / f"{img_id}.jpg")), cv2.COLOR_BGR2RGB)
    raw = cv2.imread(str(root / "masks" / f"{img_id}.jpg"), cv2.IMREAD_GRAYSCALE)
    return img, raw > 127, raw

img, mask, raw = load_sample(ids[0])
print("immagine:", img.shape, img.dtype, "| maschera:", mask.shape, mask.dtype)
print("valori distinti nella maschera grezza:", len(np.unique(raw)), "→ dopo binarizzazione:", np.unique(mask))
print("frazione di pixel polipo:", round(mask.mean(), 3))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(img); ax[0].set_title("immagine")
ax[1].imshow(mask, cmap="gray"); ax[1].set_title("maschera GT")
for a in ax: a.axis("off")

## 5. Il box prompt

Il box è il rettangolo più piccolo che contiene il polipo, nel formato di SAM `[x0, y0, x1, y1]`.
Se nell'immagine ci sono più polipi, questo box li racchiude tutti (box "unione"): per ora va bene così.

In [ ]:
from matplotlib.patches import Rectangle

def box_from_mask(mask):
    ys, xs = np.where(mask)
    return np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=np.float32)

def draw_box(ax, box, color="lime"):
    x0, y0, x1, y1 = box
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, linewidth=2))

box = box_from_mask(mask)
print("box:", box)

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(img); draw_box(ax, box); ax.axis("off")

## 6. Prima predizione di SAM

`SamPredictor` fa internamente tutto il preprocessing (resize del lato lungo a 1024, normalizzazione, padding) e restituisce la maschera alla risoluzione originale.
Nel progetto scriveremo noi questi passaggi, perché durante il training serviranno in batch e con i gradienti.

- `set_image` esegue l'**image encoder** (la parte pesante, una volta per immagine).
- `predict` esegue **prompt encoder + mask decoder** (leggeri, si possono chiamare più volte con prompt diversi).

In [ ]:
import time
from segment_anything import sam_model_registry, SamPredictor

device = "cuda"
sam = sam_model_registry["vit_b"](checkpoint=str(CKPT)).to(device).eval()
predictor = SamPredictor(sam)

n_enc = sum(p.numel() for p in sam.image_encoder.parameters())
n_dec = sum(p.numel() for p in sam.mask_decoder.parameters())
print(f"parametri image encoder: {n_enc/1e6:.1f}M | mask decoder: {n_dec/1e6:.1f}M")

t = time.time()
predictor.set_image(img)
t_enc = time.time() - t
pred_masks, scores, _ = predictor.predict(box=box, multimask_output=False)
pred = pred_masks[0]
print(f"encoder: {t_enc:.2f}s | maschera predetta: {pred.shape} {pred.dtype} | score IoU stimato: {scores[0]:.3f}")

## 7. Dice e IoU

- **Dice** = 2·|P∩G| / (|P| + |G|)
- **IoU** = |P∩G| / |P∪G|

Entrambi valgono 1 con sovrapposizione perfetta e 0 con maschere disgiunte. Il Dice è sempre ≥ IoU.

In [ ]:
def dice(pred, gt, eps=1e-8):
    inter = np.logical_and(pred, gt).sum()
    return float(2 * inter / (pred.sum() + gt.sum() + eps))

def iou(pred, gt, eps=1e-8):
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return float(inter / (union + eps))

# controllo minimo: casi di cui conosciamo la risposta
a = np.zeros((10, 10), bool); a[2:5, 2:5] = True
b = np.zeros((10, 10), bool); b[6:9, 6:9] = True
assert abs(dice(a, a) - 1) < 1e-6 and dice(a, b) == 0.0

print(f"Dice: {dice(pred, mask):.3f} | IoU: {iou(pred, mask):.3f}")

def show(img, gt, pred, box, title=""):
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ax[0].imshow(img); draw_box(ax[0], box); ax[0].set_title("immagine + box")
    ax[1].imshow(gt, cmap="gray"); ax[1].set_title("GT")
    ax[2].imshow(pred, cmap="gray"); ax[2].set_title(f"SAM {title}")
    for a_ in ax: a_.axis("off")
    plt.show()

show(img, mask, pred, box, f"Dice={dice(pred, mask):.3f}")

## 8. Box "realistico" con jitter

Il box perfetto è un aiuto irrealistico. Il jitter sposta ogni coordinata di una frazione casuale della larghezza/altezza del box, come farebbe un medico che lo disegna a mano.
Il generatore ha un seed fisso: stesso seed → stesso box.

In [ ]:
def jitter_box(box, ratio, rng, h, w):
    bw, bh = box[2] - box[0], box[3] - box[1]
    noise = rng.uniform(-ratio, ratio, 4) * np.array([bw, bh, bw, bh])
    return np.clip(box + noise, 0, [w - 1, h - 1, w - 1, h - 1]).astype(np.float32)

h, w = mask.shape
for ratio in [0.0, 0.05, 0.1, 0.2]:
    jb = jitter_box(box, ratio, np.random.default_rng(0), h, w)
    p = predictor.predict(box=jb, multimask_output=False)[0][0]   # set_image già fatto: niente encoder
    print(f"jitter {ratio:.2f} → Dice {dice(p, mask):.3f}")

## 9. Primo Dice medio su 20 immagini

Non è ancora il risultato zero-shot ufficiale: quello lo calcoleremo sullo split di test fisso (120 immagini).
Serve per vedere la variabilità tra immagini e i casi difficili.

In [ ]:
import random

sample_ids = random.Random(0).sample(ids, 20)
rows = []
for img_id in sample_ids:
    im, gt, _ = load_sample(img_id)
    bx = box_from_mask(gt)
    predictor.set_image(im)
    pm = predictor.predict(box=bx, multimask_output=False)[0][0]
    pj = predictor.predict(box=jitter_box(bx, 0.1, np.random.default_rng(0), *gt.shape),
                           multimask_output=False)[0][0]
    rows.append((img_id, dice(pm, gt), dice(pj, gt), gt.mean()))

d_gt = np.array([r[1] for r in rows]); d_j = np.array([r[2] for r in rows])
print(f"Dice medio box GT:      {d_gt.mean():.3f} ± {d_gt.std():.3f}")
print(f"Dice medio box jitter:  {d_j.mean():.3f} ± {d_j.std():.3f}")

worst = sorted(rows, key=lambda r: r[1])[:3]
print("casi peggiori:", [(r[0], round(r[1], 3), f"area={r[3]:.2f}") for r in worst])

## 10. Guarda i casi peggiori

In [ ]:
for img_id, d, _, _ in worst:
    im, gt, _ = load_sample(img_id)
    bx = box_from_mask(gt)
    predictor.set_image(im)
    pm = predictor.predict(box=bx, multimask_output=False)[0][0]
    show(im, gt, pm, bx, f"Dice={d:.3f}")

## Da annotare prima del prossimo step

1. Tempo di `set_image` per immagine sulla tua GPU (ti serve per stimare la durata di un'epoca).
2. Dice medio con box GT e con jitter: quanto pesa il jitter?
3. Nei casi peggiori, cosa sbaglia SAM? (bordi poco contrastati, più polipi nello stesso box, riflessi della luce...)
4. Prova `multimask_output=True`: SAM restituisce 3 maschere con 3 score. Quale sceglieresti, e come?

Queste osservazioni sono materiale per il README e per la presentazione su SAM-Adapter.